In [25]:
from pathlib import Path
import random
import shutil
from collections import Counter

In [26]:
DATASET_DIR = Path("../dataset/tilda defect ntu")
OUTPUT_DIR = Path("../dataset/tilda defect balanced")

print("Dataset path :", DATASET_DIR)
print("Output path  :", OUTPUT_DIR)

Dataset path : ..\dataset\tilda defect ntu
Output path  : ..\dataset\tilda defect balanced


In [27]:
for split in ["train", "valid", "test"]:
    split_dir = DATASET_DIR / split
    print(f"\n{split.upper()}")
    print("Images :", (split_dir / "images").exists())
    print("Labels :", (split_dir / "labels").exists())


TRAIN
Images : True
Labels : True

VALID
Images : True
Labels : True

TEST
Images : True
Labels : True


In [28]:
CLASS_NAMES = {
    0: "hole",
    1: "objects",
    2: "oil spot",
    3: "thread error"
}

TARGET_PER_CLASS = 200

print("Jumlah kelas:", len(CLASS_NAMES))

for class_id, class_name in CLASS_NAMES.items():
    print(f"{class_id}: {class_name}")

Jumlah kelas: 4
0: hole
1: objects
2: oil spot
3: thread error


In [29]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

all_data = []

for split in ["train", "valid", "test"]:
    image_dir = DATASET_DIR / split / "images"
    label_dir = DATASET_DIR / split / "labels"

    for image_path in image_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        all_data.append({
            "image": image_path,
            "label": label_path,
            "original_split": split
        })

print("Total pasangan image + label:", len(all_data))

Total pasangan image + label: 896


In [30]:
original_split_counts = Counter(
    item["original_split"]
    for item in all_data
)

print("Distribusi dataset asli:\n")

for split in ["train", "valid", "test"]:
    print(f"{split:5} : {original_split_counts[split]}")

Distribusi dataset asli:

train : 776
valid : 80
test  : 40


In [31]:
data_by_class = {
    class_id: []
    for class_id in CLASS_NAMES
}

multi_class_images = []
unknown_class_images = []

for item in all_data:
    label_path = item["label"]

    with open(label_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    class_ids = set()

    for line in lines:
        parts = line.strip().split()

        if len(parts) == 0:
            continue

        class_id = int(parts[0])

        if class_id in CLASS_NAMES:
            class_ids.add(class_id)
        else:
            unknown_class_images.append(item)

    if len(class_ids) == 1:
        class_id = next(iter(class_ids))
        data_by_class[class_id].append(item)

    elif len(class_ids) > 1:
        multi_class_images.append(item)

print("Proses pembacaan label selesai.")
print("Image dengan lebih dari satu kelas:", len(multi_class_images))
print("Image dengan class ID tidak dikenal:", len(unknown_class_images))

Proses pembacaan label selesai.
Image dengan lebih dari satu kelas: 0
Image dengan class ID tidak dikenal: 0


In [32]:
print("Jumlah image setiap kelas:\n")
for class_id, class_name in CLASS_NAMES.items():
    count = len(data_by_class[class_id])
    print(f"{class_name:15} : {count}")

Jumlah image setiap kelas:

hole            : 219
objects         : 235
oil spot        : 219
thread error    : 223


In [33]:
print("Target balancing:\n")

for class_id, class_name in CLASS_NAMES.items():
    original_count = len(data_by_class[class_id])
    print(
        f"{class_name:15} : "
        f"{original_count} -> {TARGET_PER_CLASS}"
    )

Target balancing:

hole            : 219 -> 200
objects         : 235 -> 200
oil spot        : 219 -> 200
thread error    : 223 -> 200


In [34]:
random.seed(42)
balanced_data = {}

for class_id, class_name in CLASS_NAMES.items():
    class_data = data_by_class[class_id].copy()
    # Acak data
    random.shuffle(class_data)
    # Ambil 200 image
    selected_data = class_data[:TARGET_PER_CLASS]
    balanced_data[class_id] = selected_data

    print(
        f"{class_name:15} : "
        f"{len(selected_data)} images"
    )

hole            : 200 images
objects         : 200 images
oil spot        : 200 images
thread error    : 200 images


In [35]:
# Jika folder output sudah ada dari percobaan sebelumnya,
# hapus terlebih dahulu agar hasilnya bersih.

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# Buat struktur folder
for split in ["train", "valid", "test"]:

    (OUTPUT_DIR / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT_DIR / split / "labels").mkdir(
        parents=True,
        exist_ok=True
    )

print("Folder dataset balanced berhasil dibuat.")

Folder dataset balanced berhasil dibuat.


In [36]:
TRAIN_RATIO = 0.70
VALID_RATIO = 0.20
TEST_RATIO = 0.10

TRAIN_PER_CLASS = int(TARGET_PER_CLASS * TRAIN_RATIO)
VALID_PER_CLASS = int(TARGET_PER_CLASS * VALID_RATIO)
TEST_PER_CLASS = int(TARGET_PER_CLASS * TEST_RATIO)

print("Pembagian per kelas:")
print("Train      :", TRAIN_PER_CLASS)
print("Validation :", VALID_PER_CLASS)
print("Test       :", TEST_PER_CLASS)

print("\nTotal:")
print("Train      :", TRAIN_PER_CLASS * 4)
print("Validation :", VALID_PER_CLASS * 4)
print("Test       :", TEST_PER_CLASS * 4)

Pembagian per kelas:
Train      : 140
Validation : 40
Test       : 20

Total:
Train      : 560
Validation : 160
Test       : 80


In [37]:
for class_id, class_name in CLASS_NAMES.items():
    data = balanced_data[class_id]

    train_data = data[:140]
    valid_data = data[140:180]
    test_data = data[180:200]

    split_data = {
        "train": train_data,
        "valid": valid_data,
        "test": test_data
    }

    for split, items in split_data.items():
        for item in items:
            image_path = item["image"]
            label_path = item["label"]

            destination_image = (OUTPUT_DIR/split/"images"/image_path.name)
            destination_label = (OUTPUT_DIR/split/"labels"/label_path.name)
            shutil.copy2(image_path, destination_image)
            shutil.copy2(label_path, destination_label)

print("Proses splitting dan copying selesai.")

Proses splitting dan copying selesai.


In [38]:
print("HASIL DATASET BALANCED\n")

for split in ["train", "valid", "test"]:
    image_count = len(list((OUTPUT_DIR / split / "images").glob("*")))
    label_count = len(list((OUTPUT_DIR / split / "labels").glob("*.txt")))

    print(
        f"{split.upper():5} : "
        f"{image_count} images & "
        f"{label_count} labels"
    )

HASIL DATASET BALANCED

TRAIN : 560 images & 560 labels
VALID : 160 images & 160 labels
TEST  : 80 images & 80 labels


In [39]:
print("DISTRIBUSI KELAS")

for split in ["train", "valid", "test"]:
    class_counter = Counter()
    label_dir = OUTPUT_DIR / split / "labels"

    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r", encoding="utf-8") as file:

            for line in file:
                parts = line.strip().split()

                if len(parts) > 0:
                    class_id = int(parts[0])
                    class_counter[class_id] += 1

    print(f"\n{split.upper()}")

    for class_id, class_name in CLASS_NAMES.items():
        print(
            f"{class_name:15} : "
            f"{class_counter[class_id]}"
        )

DISTRIBUSI KELAS

TRAIN
hole            : 155
objects         : 176
oil spot        : 163
thread error    : 183

VALID
hole            : 45
objects         : 52
oil spot        : 45
thread error    : 50

TEST
hole            : 21
objects         : 24
oil spot        : 22
thread error    : 25


In [40]:
yaml_content = f"""path: {OUTPUT_DIR.as_posix()}
train: train/images
val: valid/images
test: test/images

nc: 4

names:
  0: hole
  1: objects
  2: oil spot
  3: thread error
"""

yaml_path = OUTPUT_DIR / "data-balanced.yaml"

with open(yaml_path, "w", encoding="utf-8") as file:
    file.write(yaml_content)

print("data-balanced.yaml berhasil dibuat:")
print(yaml_path)

data-balanced.yaml berhasil dibuat:
..\dataset\tilda defect balanced\data-balanced.yaml


In [41]:
print(yaml_path.read_text(encoding="utf-8"))

path: ../dataset/tilda defect balanced
train: train/images
val: valid/images
test: test/images

nc: 4

names:
  0: hole
  1: objects
  2: oil spot
  3: thread error



In [42]:
print("FINAL DATASET SUMMARY")

print(f"Total classes       : {len(CLASS_NAMES)}")
print(f"Images per class    : {TARGET_PER_CLASS}")
print(f"Total images        : {TARGET_PER_CLASS * len(CLASS_NAMES)}")

print("\nSplit:")
print(f"Train               : {TRAIN_PER_CLASS * 4}")
print(f"Validation          : {VALID_PER_CLASS * 4}")
print(f"Test                : {TEST_PER_CLASS * 4}")

print("\nPer class:")
for class_id, class_name in CLASS_NAMES.items():
    print(
        f"- {class_name:15}: "
        f"{TARGET_PER_CLASS} images"
    )

print("\nDataset location:")
print(OUTPUT_DIR)

FINAL DATASET SUMMARY
Total classes       : 4
Images per class    : 200
Total images        : 800

Split:
Train               : 560
Validation          : 160
Test                : 80

Per class:
- hole           : 200 images
- objects        : 200 images
- oil spot       : 200 images
- thread error   : 200 images

Dataset location:
..\dataset\tilda defect balanced
